In [ ]:
# Example 1: create an in-memory repo, write a zarr array in a session, commit.
# Source: icechunk.io getting-started/quickstart (adapted, Apache-2.0).
# Modified: in-memory storage, small arrays, and assertions on the results.
import icechunk
import numpy as np
import zarr

repo = icechunk.Repository.create(icechunk.in_memory_storage())
session = repo.writable_session("main")

root = zarr.group(store=session.store)
arr = root.create_array("temperature", shape=(4, 4), chunks=(2, 2), dtype="float32")
arr[:] = np.arange(16, dtype="float32").reshape(4, 4)

snapshot_id = session.commit("write initial temperature field")
print("committed snapshot:", snapshot_id)
assert session.read_only  # a committed writable session becomes read-only
print("post-commit session is read-only, as documented")

In [ ]:
# Example 2: time travel — read an old snapshot after overwriting on main.
# Source: icechunk.io getting-started/quickstart (adapted, Apache-2.0).
# Modified: in-memory storage, small arrays, and assertions on the results.
import icechunk
import numpy as np
import zarr

repo = icechunk.Repository.create(icechunk.in_memory_storage())
s1 = repo.writable_session("main")
zarr.group(store=s1.store).create_array("t", shape=(4,), chunks=(4,), dtype="float32")[:] = np.arange(4, dtype="float32")
snap1 = s1.commit("v1: 0..3")

s2 = repo.writable_session("main")
zarr.open_group(s2.store, mode="r+")["t"][:] = 100.0
s2.commit("v2: all 100s")

old = repo.readonly_session(snapshot_id=snap1)
assert old.read_only
old_t = zarr.open_group(old.store, mode="r")["t"][:]
new_t = zarr.open_group(repo.readonly_session(branch="main").store, mode="r")["t"][:]
assert old_t.tolist() == [0.0, 1.0, 2.0, 3.0] and new_t.tolist() == [100.0] * 4
print("time travel OK — snapshot", snap1, "still serves", old_t)

In [ ]:
# Example 3: branches + tags — create, list, look up, and open sessions on them.
# Source: icechunk.io understanding/version-control (adapted, Apache-2.0).
# Modified: in-memory storage, small arrays, and assertions on the results.
import icechunk
import numpy as np
import zarr

repo = icechunk.Repository.create(icechunk.in_memory_storage())
with repo.transaction("main", message="seed") as store:
    zarr.group(store=store).create_array("x", shape=(2,), chunks=(2,), dtype="int32")[:] = np.array([1, 2], "int32")
snap = repo.lookup_branch("main")

repo.create_branch("dev", snapshot_id=snap)
repo.create_tag("v1.0.0", snapshot_id=snap)
print("branches:", repo.list_branches(), "tags:", repo.list_tags())
assert {"main", "dev"} <= repo.list_branches() and "v1.0.0" in repo.list_tags()
assert repo.lookup_branch("dev") == snap

tagged = zarr.open_group(repo.readonly_session(tag="v1.0.0").store, mode="r")
assert tagged["x"][:].tolist() == [1, 2]
print("branch/tag round trip OK — tag v1.0.0 resolves to", snap)

In [ ]:
# Example 4: transaction context manager + ancestry — history is newest-first
# and carries the commit messages, ending at the repo-init snapshot.
# Source: icechunk.io getting-started/quickstart (adapted, Apache-2.0).
# Modified: in-memory storage, small arrays, and assertions on the results.
import icechunk
import numpy as np
import zarr

repo = icechunk.Repository.create(icechunk.in_memory_storage())
with repo.transaction("main", message="write field") as store:
    zarr.group(store=store).create_array("f", shape=(4,), chunks=(4,), dtype="float64")[:] = np.full(4, 8.0)
with repo.transaction("main", message="halve the field") as store:
    g = zarr.open_group(store, mode="r+")
    g["f"][:] = g["f"][:] / 2.0

history = list(repo.ancestry(branch="main"))
for ancestor in history:
    print(" ", ancestor.id, repr(ancestor.message), ancestor.written_at)
assert [a.message for a in history[:2]] == ["halve the field", "write field"]
final = zarr.open_group(repo.readonly_session(branch="main").store, mode="r")["f"][:]
assert final.tolist() == [4.0] * 4
print("transaction + ancestry OK — history depth", len(history))

In [ ]:
# Example 5: concurrent writers — conflict detection on commit + rebase with a
# chunk-conflict solver (the headline "transactional zarr" guarantee).
# Source: icechunk.io version-control conflict docs (adapted, Apache-2.0).
# Modified: in-memory storage, small arrays, and assertions on the results.
import icechunk
import numpy as np
import zarr

repo = icechunk.Repository.create(icechunk.in_memory_storage())
with repo.transaction("main", message="seed") as store:
    zarr.group(store=store).create_array("x", shape=(4,), chunks=(4,), dtype="int32")[:] = np.zeros(4, "int32")

s1 = repo.writable_session("main")
s2 = repo.writable_session("main")
zarr.open_group(s1.store, mode="r+")["x"][:] = 1
zarr.open_group(s2.store, mode="r+")["x"][:] = 2
s1.commit("writer 1")
try:
    s2.commit("writer 2")
    raise AssertionError("second writer must conflict")
except icechunk.ConflictError as e:
    print("ConflictError as expected:", str(e)[:70])

s2.rebase(icechunk.BasicConflictSolver(on_chunk_conflict=icechunk.VersionSelection.UseOurs))
s2.commit("writer 2 after rebase")
final = zarr.open_group(repo.readonly_session(branch="main").store, mode="r")["x"][:]
assert final.tolist() == [2, 2, 2, 2]
print("rebase resolved the chunk conflict; final =", final.tolist())